In [7]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"lukealley","key":"8084d6e800acf53ed29302ae27d43b41"}'}

In [8]:
import os, shutil

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

print("Kaggle ready")

Kaggle ready


In [9]:
!kaggle datasets download -d andrewmvd/face-mask-detection
!unzip -q face-mask-detection.zip

print("Dataset downloaded")

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/face-mask-detection
License(s): CC0-1.0
100% 398M/398M [00:22<00:00, 18.8MB/s]

Dataset downloaded


In [10]:
import os
import cv2
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [11]:
images_dir = "images"
annotations_dir = "annotations"

output_dir = "mask_data"
classes = ["with_mask", "without_mask", "mask_weared_incorrect"]

for c in classes:
    os.makedirs(os.path.join(output_dir, c), exist_ok=True)

count = 0

for xml_file in os.listdir(annotations_dir):
    if not xml_file.endswith(".xml"):
        continue

    tree = ET.parse(os.path.join(annotations_dir, xml_file))
    root = tree.getroot()

    filename = root.find("filename").text
    img = cv2.imread(os.path.join(images_dir, filename))

    if img is None:
        continue

    for obj in root.findall("object"):
        label = obj.find("name").text
        bbox = obj.find("bndbox")

        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        face = img[ymin:ymax, xmin:xmax]

        if face.size == 0:
            continue

        cv2.imwrite(f"{output_dir}/{label}/{count}.jpg", face)
        count += 1

print("Done:", count)

Done: 4072


In [12]:
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train = datagen.flow_from_directory(
    "mask_data",
    target_size=(128,128),
    subset="training"
)

val = datagen.flow_from_directory(
    "mask_data",
    target_size=(128,128),
    subset="validation"
)

Found 3259 images belonging to 3 classes.
Found 813 images belonging to 3 classes.


In [13]:
model = Sequential([
    Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
    MaxPooling2D(),
    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(64,activation='relu'),
    Dense(3,activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
model.fit(train, validation_data=val, epochs=5)

Epoch 1/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 82s 785ms/step - accuracy: 0.8898 - loss: 0.3332 - val_accuracy: 0.9373 - val_loss: 0.1733
Epoch 2/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 74s 726ms/step - accuracy: 0.9331 - loss: 0.2123 - val_accuracy: 0.9385 - val_loss: 0.1949
Epoch 3/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 72s 710ms/step - accuracy: 0.9451 - loss: 0.1783 - val_accuracy: 0.9422 - val_loss: 0.1568
Epoch 4/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 78s 765ms/step - accuracy: 0.9457 - loss: 0.1519 - val_accuracy: 0.9397 - val_loss: 0.2064
Epoch 5/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 76s 743ms/step - accuracy: 0.9567 - loss: 0.1218 - val_accuracy: 0.9397 - val_loss: 0.1472


In [15]:
model.evaluate(val)

26/26 ━━━━━━━━━━━━━━━━━━━━ 5s 183ms/step - accuracy: 0.9397 - loss: 0.1472


[0.14724326133728027, 0.9397293925285339]